In [7]:
"""
Coder: Matteo Houston
Date (last upd): 7/15/2026

Purpose/Goal: Testing a variety of signal-processing packages and methods to determine possible best route forward. Most of the code here
was treated more as a sandbox for testing than an actual environment to use for in a pipeline. In addition, in part from being a novice coder, 
most of my functions or methods may not be as optimal or as legible as possible. Regardless, I present most of what I worked on in this 
file.

Note: I do not recommend using any method with "# - WIP -". It likely will not work or work as intended.
Note: You can find the descriptions for each methods by using the in-built python method 'help(something)'.

Available methods:
    - pull_real_gw()
    - pull_sim_gw()
    - quick_process()
    - fft_it()
    - cwt_it()
    - stft_it()
    - reconstruct_it() #WIP
    - rmse() #WIP
    - get_sim_snr() #WIP
    - get_best_n() #WIP
    - performance() #WIP
"""

'\nCoder: Matteo Houston\nDate (last upd): 7/15/2026\n\nPurpose/Goal: Testing a variety of signal-processing packages and methods to determine possible best route forward. Most of the code here\nwas treated more as a sandbox for testing than an actual environment to use for in a pipeline. In addition, in part from being a novice coder, \nmost of my functions or methods may not be as optimal or as legible as possible. Regardless, I present most of what I worked on in this \nfile.\n\nNote: I do not recommend using any method with "# - WIP -". It likely will not work or work as intended.\nNote: You can find the descriptions for each methods by using the in-built python method \'help(something)\'.\n\nAvailable methods:\n    - pull_real_gw()\n    - pull_sim_gw()\n    - quick_process()\n    - fft_it()\n    - cwt_it()\n    - stft_it()\n    - reconstruct_it() #WIP\n    - rmse() #WIP\n    - get_sim_snr() #WIP\n    - get_best_n() #WIP\n    - performance() #WIP\n'

In [8]:
### Import packages.

In [9]:
# -- ignore the warning if it pops up, haven't had any issues because of it. -- #
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ssqueezepy
from ssqueezepy import ssq_cwt, issq_cwt, extract_ridges

# from scipy import signal
from scipy import integrate
from gwpy.timeseries import TimeSeries
from gwosc.datasets import event_gps

In [10]:
def pull_real_gw(eventname = "", detectorsite = "", duration = 4):
    """
    Function to pull a given gravity wave by its name for a given duration surrounding the chirp, 
    e.g. GW150914 from the Hanford Detector Site for four seconds, two seconds before and after the chirp.

    Parameters:
        eventname (String)    - event name with GW designation, pulled from GWpy
        detectorsite (String) - 'H1' for hanford, 'L1' for Livingston. Virgo isn't working at the time of writing this. See GWpy docs.
        duration (Int)        - in seconds, the duration surrounding the chirp, by default 4.

    Returns:
        data (TimeSeries)     - pulled gravity wave from GWosc/GWpy
        
    """
    gps = event_gps(eventname)
    det = detectorsite
    dur = duration

    # duration is total time surrounding both sides of event; i.e., default is 2 seconds before and 2 seconds after
    data = TimeSeries.fetch_open_data(det, gps - (dur/2), gps + (dur/2))

    return data

In [11]:
def pull_sim_gw(signalfile = "", noisefile = "", customfreq = 4096):
    """
    Works with .CSV files. Joins any simulated signal file to a noise file, given that the length of the signalfile is less than the 
    length of the noise file. This function will join the signal file in the noise file at the center-point of the noise file.

    Parameters
        signalfile (String) - directory to the generated .csv file of the gravitational wave.
        noisefile (String)  - directory to the generated .csv file for the noise model.
        customfreq (Int)    - in Hz, the given frequency used to generate the noise and signals. Not sure why I didn't just automate this.
        
    Returns:
        data (TimeSeries)   - the combined signal and noise files
    """
    sig = signalfile
    ns = noisefile

    # -- first using pandas to read the .csv files because pandas is better with .csv. not sure about gwf or ndf5 files. -- #
    # -- might want to update this function to allow for other file formats for sim data -- #
    
    # this is the signal array
    sigf = pd.read_csv(
        sig,
        sep=None,
        engine="python",
        skiprows=1,
        header=None,
        names=["time", "hplus"]
        )

    # this is the noise array
    nsf = pd.read_csv(
        ns,
        sep = None,
        engine = "python",
        skiprows= 1,
        header = None,
        names=["time", "strain"]
        )
    
    if customfreq != 4096: # set at 4096. this is the frequency that i worked at for the most part. not sure why im not automating this.
        dt = customFreq
    
    signal = TimeSeries(
        sigf["hplus"].values,
        t0 = -1 * sigf["time"].iloc[0], # start time
        dt = 1/4096,                    # time step
        unit="",
        name="h+"
    )
    noise = TimeSeries(
        nsf["strain"].values,
        t0 = nsf["time"].iloc[0], # start time
        dt = 1/4096,              # time step
        unit = "",
        name = "strain"
    )

    midpoint = int((len(noise)/2 - 1))            # find midpoint
    startindex = int(abs(midpoint - len(signal))) # start index within noise file to add signal onto

    for i in range(len(signal)):
        noise[startindex + i] += signal[i]

    data = noise
    return data

In [12]:
"""
One of the first stages of signal processing is taking the signal and "whitening" and bandpassing the noise. This is to get rid of the big noise 
sources, mainly low-frequency noises that originate from electronic, anthropogenic, and geologic sources.
"""

'\nOne of the first stages of signal processing is taking the signal and "whitening" and bandpassing the noise. This is to get rid of the big noise \nsources, mainly low-frequency noises that originate from electronic, anthropogenic, and geologic sources.\n'

In [13]:
def quick_process(input_signal, fftlength = 2, low_frequency = 35, high_frequency = 350):
    """
    The initial steps typically taken by the LIGO/VIRGO collaboration to break down a signal before applying various transforms.
    Could be useful if our only goal is to identify a signal. 

    Parameters:
        input_signal (TimeSeries) - either the combined simulated GW, or the pulled GW from GWpy.
        fftlength (Int)       - the length to which fft over, default 2
        low_frequency (Int)   - in Hz, the low-end of the frequency to bandpass, default 35
        high_frequency (Int)  - in Hz, the high-end of the frequency to bandpass, default 350
        
    Returns: 
        data (TimeSeries)     - whitened and bandpassed data
    """
    inp = input_signal
    lf = low_frequency
    hf = high_frequency

    white = inp.whiten(fftlength, window = ("tukey", 0.2))
    bp = white.bandpass(lf, hf)
    
    data = bp
    return data

In [14]:
def fft_it(input_data, preprocess = False):
    """
    Basic fast Fourier transform, using numpy's. For the future, I'd try messing with FFTW, which seems to be a common
    theme amongst the other packages. I only used numpy because I was struggling to get *only* the real part of the FFT.

    Parameters:
        input_data (TimeSeries) - the simulated/real GW.
        preprocess (Bool)       - calls the quick_process method to whiten and bandpass the data. Now come to think of it, I need to fix this
                                  to allow input_data for various lengths and sizes.
                                  
    Returns:
        fft_data (numpy array)  - fft'd data
    """
    data = input_data
    if preprocess:
        data = quick_process(data)
    data = np.array(data)        # numpy cannot read TimeSeries objects
    fft_data = np.fft.rfft(data) # using numpy's fft method. recommend checking others
    return fft_data

In [15]:
def cwt_it(input_data, preprocess = False, plot = False):
    """
    A basic implementation of the continuous wavelet transform as provided by the ssqueezepy package. Works... for the most part.

    Parameters:
        input_data (TimeSeries) - the simulated/real GW
        preprocess (Bool)       - whether to send to quick_process or not
        plot (Bool)             - to plot or not to plot

    Returns:
        Wx (numpy array)              - returns the transformed data

    """
    data = input_data
    if preprocess:
        data = quick_process(input_data)
    fs = 4096
    wavelet = ssqueezepy.wavelets.Wavelet('gmw')  # would recommend playing with various wavelets. read ssqueezepy docs.
    Wx, scales = ssqueezepy.cwt(data, wavelet, scales='log', fs=fs)
    freqs = ssqueezepy.experimental.scale_to_freq(scales, wavelet, N=len(data), fs=fs)
    t = np.arange(len(data)) / fs
    if plot: 
        plt.pcolormesh(t, freqs, np.abs(Wx), cmap='turbo', shading='auto')
        plt.yscale('log'); plt.xlabel('Time [s]'); plt.ylabel('Frequency [Hz]')
    return Wx

In [16]:
# WIP(?), code partially generated by Claude #
def stft_it(input_data, nfft = 512, low = 30, high = 350, preprocess = False, plot = False, absv = True, rdv = False):
    """
    Basic short time fast Fourier transform, using ssqueezepy. For the future, I'd mess with a LOT of ssqueezepy's methods. They seem quite
    promising, also considering the package itself is curated towards optimization. 

    Parameters:
        input_data (TimeSeries) - the simulated/real GW
        preprocess (Bool)       - calls the quick_process method to whiten and bandpass the data. 
        nfft (Int)              - the number of FFT's for a window (I think? just see ssqueezepy's docs on this), default at 512
        low (Int)               - in Hz, the low-end of the bandpass, default 30
        high (Int)              - in Hz, the high-end of the badnpass, default 350
        plot (Bool)             - to plot or not to plot
        absv (Bool)             - to take the absolute value of the STFT when plotting, default at True
        rdv (Bool)              - to display the ridges of the STFT when plotting, default at False
        
    Returns:
        stft_data (numpy array) - stft'd data
    """
    fs = 4096
    data = input_data
    if preprocess:
        data = quick_process(input_data)
    ab = 0
    rd = 0
    Sx = ssqueezepy.stft(data)                    # have not had the chance to mess with other paramters of the
                                                  # stft function from ssqueezepy; would recommend messing with it more
    if plot:
            freqs = np.fft.rfftfreq(nfft, 1/fs)   
            band = (freqs >= low) & (freqs <= high) # your bandpass range
            t = np.arange(Sx.shape[1]) / fs 
            if absv == True: # doing this because, for wtv reason, imshow from ssqueezepy takes 1 or 0 instead of just Bool values
                ab = 1
            if rdv == True:
                rd = 1
            plt.figure(figsize = (12,4))
            ssqueezepy.visuals.imshow(Sx[band], xticks = t, abs = ab, ridge = rd)
    stft_data = Sx
    return stft_data

In [17]:
# WIP, code partially generated by Claude #
def reconstruct_it(fft_coeffs, n_keep=1):
    """
    - WIP -
    Takes highest bins of an FFT to reconstruct the signal. I don't think this works as well as we imagine it to.

    Parameters:
        fft_coeffs (numpy array) - fft'd data
        n_keep (Int)             - number of bins to keep

    Returns:
        data (numpy array)       - reconstructed signal
    - WIP -
    """
    
    # the code below works; might try something else in the future (see ssqueezepy's ridge extraction and reconstruction?)
    coeffs = np.array(fft_coeffs)
    keep = np.argsort(np.abs(coeffs))[-n_keep:]   # bins of the n_keep largest coeffs
    filtered = np.zeros_like(coeffs)              # full-length zero spectrum
    filtered[keep] = coeffs[keep]                 # restore FULL complex coeff (phase intact)
    data = np.fft.ifft(filtered).real             # real-valued reconstructed signal
   
    return data

In [18]:
# WIP, VERY unoptimized :( 
def rmse(predicted, actual, graph = False):
    """
    - WIP -
    Calculates the RMSE of the reconstructed signal against the actual signal. I don't think this works as intended.

    Parameters:
        predicted (TimeSeries) - predicted chirp (reconstructed, I believe)
        actual (TimeSeries)    - regular chirp.
        graph (Bool)           - whether to graph or not, default at False

    Returns:
        data (numpy array)     - rmse value
    - WIP -
    """
    pre = predicted
    act = actual
    N = pre.size         # bin size
    S = 0                # summation var
    ylabel = "RMSE"
    
    for i in range(N):
        S += (pre[i] - act[i]) ** 2
    S /= N
    data = S ** 0.5
    
    if graph == False:
        return data
    else:
        a = [0] * new_data.size
        for i in range(new_data.size):
            recon = reconstruct_it(fft_it(act), i)
            RMSE = rmse(recon, act)
            print(f"For n = {i}, RMSE = " f"{RMSE}")
            a[i] = RMSE
        plt.figure(figsize = (12, 4))
        plt.xlabel("number of bins")
        plt.ylabel(ylabel)
        plt.plot(a, 'r+');
        return data

In [19]:
def get_sim_snr(chirpfile, noisefile, minf = 35, maxf = 2048, fftlength = 4):

    sig = chirpfile 
    ns = noisefile

    """
    Given a known noise model from simulated data (from LALsuite), calculate the SNR. Seems to be accurate, but probably not realistic, 
    if only because we alread know exactly what our noise model is (for example, using the ideal binary black hole at 20 deg from LALsuite).

    Parameters:
        chirpfile (String) - directory to some csv file
        noisefile (String) - directory to some csv file
        minf (Int)         - minimum frequency, default at 35 Hz
        maxf (Int)         - maximum frequency, default at nyquist frequency (max frequency / 2).
        fftlength (Int)    - length to perform fft over

    Returns:
        snr_value (float)  - returns given SNR for the simulated data
    """
    # yes i just copied these over from grab sim gw
    sigf = pd.read_csv(
        sig,
        sep=None,
        engine="python",
        skiprows=1,
        header=None,
        names=["time", "hplus"]
        )
    nsf = pd.read_csv(
        ns,
        sep = None,
        engine = "python",
        skiprows= 1,
        header = None,
        names=["time", "strain"]
        )
    #################################################
    signal = TimeSeries(
        sigf["hplus"].values,
        t0 = -1 * sigf["time"].iloc[0], # start time
        dt = 1/4096,                    # time step
        unit="",
        name="h+"
    )
    noise = TimeSeries(
        nsf["strain"].values,
        t0 = nsf["time"].iloc[0],   # start time
        dt = 1/4096,                # time step
        unit = "",
        name = "strain"
    )

    # - segments below generated by Claude - #
    # fft of the clean chirp, continuous-ft convention  
    dt = signal.dt.value
    hf = np.fft.rfft(signal.value) * dt            
    freqs = np.fft.rfftfreq(len(signal), dt)

    # psd of the noise, then interpolate onto the signal's grid
    psd = noise.psd(fftlength=fftlength, overlap=fftlength/2)   
    Sn = np.interp(freqs, psd.frequencies.value, psd.value)

    
    # band-limit and integrate
    mask = (freqs >= minf) & (Sn > 0)
    integrand = 4.0 * np.abs(hf[mask])**2 / Sn[mask]

    snr_value = np.sqrt(integrate.simpson(integrand, x=freqs[mask]))
    return snr_value

In [20]:
# WIP #
def get_best_n(predicted, actual, goal = 0.01): # using RMSE, find the goal RMSE value to determine the number of fourier modes to keep
    """
    - WIP -
    This method was developed to test the idea of retrieving the highest modes from an FFT. Because the RMSE function is not particularly
    effecient, this function is not good for high values. Would recommend cleaning this RMSE up.

    Parameters:
        predicted (TimeSeries) - the reconstructed chirp
        actual (TimeSeries)    - the original chirp file
        goal (Int)             - goal RMSE to achieve, default at 0.01

    Returns:
        bestn (Int)            - the best n number of modes to keep from an FFT when reconstructing to improve effeciency
    - WIP -
    """
    pre = predicted
    act = actual
    g = goal
    bestn = 0
    for i in range(new_data.size):
        recon = reconstruct_it(fft_it(act), i + 1)
        RMSE = rmse(recon, act)
        # print(f"Current RMSE = {RMSE}")
        if RMSE <= goal:
            bestn = i + 1
            # print(f"Best RMSE at: {i + 1}!")
            break
    return bestn

In [21]:
# WIP #
def performance(graph = False): 
    """
    - WIP -
    This method was designed to return the reconstruct_it method's performance. This method is not viable for anything else.

    Parameters:
        graph (Bool) - whether to graph or not, default at False. If graphed, nothing will be returned.'
        
    Returns:
        (if graph == False), data (numpy array) - timeit results
        (if graph == True), NONE
    - WIP -
    """
    b = [0] * new_data.size
    for i in range(new_data.size):
        reconstructed = reconstructit(fftit(new_data), i)
        varResults = %timeit -o -r 10 -n 1000 reconstructit(fftit(new_data), i)
        result = varResults.average
        b[i] = result
        data = b[i]
    if graph == False:
        return data
    else:
        plt.figure(figsize = (12, 6))
        plt.xlabel("number of bins")
        plt.ylabel("Average Runtime @ 1000 loops, in Seconds")
        plt.plot(b, "bo");

In [ ]:
## create real test cases
GW150914_H1 = pull_real_gw("GW150914", "H1")
GW170817_H1 = pull_real_gw("GW170817", "H1")

In [ ]:
GW150914_L1 = pull_real_gw("GW150914", "L1")
GW170817_L1 = pull_real_gw("GW170817", "L1")

In [ ]:
## create sim test cases
simgw = pull_sim_gw("signald100.csv","constnoise.csv") ## use full directory

In [ ]:
noise = "/home/oiiwn/Desktop/srs2026/simdata/constnoise.csv"
signal = "/home/oiiwn/Desktop/srs2026/simdata/signald1000.csv"
get_sim_snr(signal, noise)

In [ ]:
## the q_transform. note that the q_transform is a method for TimeSeries objects and cannot be used for numpy arrays.

quick_process(simgw).q_transform(qrange = (4,64), frange = (35,350), whiten = True).plot(); # q_transform used directly from GWpy

In [ ]:
### the countinuous wavelet transform from ssqueezepy.

cwt_it(simgw, True, True); # need to figure out how to properly graph this. 

In [ ]:
### the short-time fourier transform from ssqueezepy.

stft_it(simgw, preprocess = True, plot = True); # i don't believe the axes are at the right scale. reaaally confused right now.

In [ ]:
"""
Reconstruction using ssqueezepy and matplotlib. This is for reference if you'd like to try ssqueezepy's reconstruction abilities for later.

"""

fs=4096
x = quick_process(GW150914_H1)

Tx, Wx, ssq_freqs, scales = ssq_cwt(x, 'gmw', fs=fs)
ridge = extract_ridges(Tx, scales=ssq_freqs, penalty=8, n_ridges=1, bw=25)

cc = ridge[:, :1]                          # (n_times, 1) ridge row-indices
cw = np.full_like(cc, 10)                  # band half-width in bins
xrec = issq_cwt(Tx, 'gmw', cc=cc, cw=cw)   # (2, n) — [component, residual]
chirp_clean = xrec[0]
t = np.arange(len(x)) / fs
residual = xrec[1]          # everything issq_cwt left out of the band

fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)

# 1. Reconstruction over whitened strain
plt.xlim(1.5,2.5)
axes[0].plot(t, x, color='0.7', lw=0.6, label='whitened strain')
axes[0].plot(t, chirp_clean, 'crimson', lw=1.0, label='ridge reconstruction')
axes[0].set_ylabel('Amplitude')
axes[0].legend(loc='upper left')

# 2. Residual — should look like featureless noise
axes[1].plot(t, residual, color='0.5', lw=0.6)
axes[1].set_ylabel('Residual')

# 3. Ridge over |Tx| — sanity-check what you inverted
axes[2].pcolormesh(t[::4], ssq_freqs, np.abs(Tx[:, ::4]),
                   cmap='turbo', shading='auto',
                   vmax=np.abs(Tx).max()*0.4)
axes[2].plot(t, ssq_freqs[cc[:, 0]], 'w', lw=1)
axes[2].set_yscale('log'); axes[2].set_ylim(20, 500)
axes[2].set_ylabel('Frequency [Hz]'); axes[2].set_xlabel('Time [s]')

fig.tight_layout()